In [6]:
import pandas as pd
import yfinance as yf
import numpy as np


In [7]:
CBA_Stock = pd.read_csv("./data/CBA_target_ready_2015_present.csv", parse_dates=["Date"])

print(CBA_Stock.head())
print(CBA_Stock.tail())
print(CBA_Stock.shape)
print(CBA_Stock.columns)


        Date       Open       High        Low      Close  Adj Close   Volume  \
0 2015-01-02  84.959686  85.277962  84.661308  85.277962  51.295269   949439   
1 2015-01-05  85.238182  85.775269  85.049202  85.486832  51.420902  1351651   
2 2015-01-06  84.641411  85.337639  84.412651  84.840332  51.032032  2477275   
3 2015-01-07  84.850281  85.029312  84.094376  84.651360  50.918358  2127190   
4 2015-01-08  85.079041  85.188446  84.671249  84.929848  51.085873  1997761   

   Target_t1  Target_t7  
0          1          0  
1          0          0  
2          0          0  
3          1          0  
4          1          0  
           Date        Open        High         Low       Close   Adj Close  \
2851 2026-04-09  180.570007  182.529999  180.470001  182.529999  182.529999   
2852 2026-04-10  181.000000  183.539993  181.000000  183.380005  183.380005   
2853 2026-04-13  183.050003  184.940002  182.619995  183.199997  183.199997   
2854 2026-04-14  185.000000  185.589996  181.94

In [8]:
CBA_Stock.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Target_t1,Target_t7
0,2015-01-02,84.959686,85.277962,84.661308,85.277962,51.295269,949439,1,0
1,2015-01-05,85.238182,85.775269,85.049202,85.486832,51.420902,1351651,0,0
2,2015-01-06,84.641411,85.337639,84.412651,84.840332,51.032032,2477275,0,0
3,2015-01-07,84.850281,85.029312,84.094376,84.651360,50.918358,2127190,1,0
4,2015-01-08,85.079041,85.188446,84.671249,84.929848,51.085873,1997761,1,0


In [9]:
CBA_Stock["Return"] = CBA_Stock["Close"].pct_change()
CBA_Stock["LogReturn"] = np.log(CBA_Stock["Close"] / CBA_Stock["Close"].shift(1))

CBA_Stock["Return_lag1"] = CBA_Stock["Return"].shift(1)
CBA_Stock["Return_lag2"] = CBA_Stock["Return"].shift(2)
CBA_Stock["Return_lag3"] = CBA_Stock["Return"].shift(3)
CBA_Stock["Return_lag5"] = CBA_Stock["Return"].shift(5)

CBA_Stock["RollingMean_5"] = CBA_Stock["Return"].rolling(5).mean()
CBA_Stock["RollingStd_5"] = CBA_Stock["Return"].rolling(5).std()

CBA_Stock["RollingMean_10"] = CBA_Stock["Return"].rolling(10).mean()
CBA_Stock["RollingStd_10"] = CBA_Stock["Return"].rolling(10).std()

CBA_Stock["VolumeChange"] = CBA_Stock["Volume"].pct_change()
CBA_Stock["VolumeMA_5"] = CBA_Stock["Volume"].rolling(5).mean()

print(CBA_Stock.shape)
print(CBA_Stock.isna().sum())

(2856, 21)
Date               0
Open               0
High               0
Low                0
Close              0
Adj Close          0
Volume             0
Target_t1          0
Target_t7          0
Return             1
LogReturn          1
Return_lag1        2
Return_lag2        3
Return_lag3        4
Return_lag5        6
RollingMean_5      5
RollingStd_5       5
RollingMean_10    10
RollingStd_10     10
VolumeChange       3
VolumeMA_5         4
dtype: int64


In [10]:
import numpy as np

window = 14

delta = CBA_Stock["Close"].diff()

gain = (delta.where(delta > 0, 0)).rolling(window).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window).mean()

rs = gain / loss
CBA_Stock["RSI_14"] = 100 - (100 / (1 + rs))

In [11]:
ema_12 = CBA_Stock["Close"].ewm(span=12, adjust=False).mean()
ema_26 = CBA_Stock["Close"].ewm(span=26, adjust=False).mean()

CBA_Stock["MACD"] = ema_12 - ema_26
CBA_Stock["MACD_signal"] = CBA_Stock["MACD"].ewm(span=9, adjust=False).mean()

In [12]:
CBA_Stock["BB_Mid"] = CBA_Stock["Close"].rolling(20).mean()
CBA_Stock["BB_Std"] = CBA_Stock["Close"].rolling(20).std()

CBA_Stock["BB_Upper"] = CBA_Stock["BB_Mid"] + 2 * CBA_Stock["BB_Std"]
CBA_Stock["BB_Lower"] = CBA_Stock["BB_Mid"] - 2 * CBA_Stock["BB_Std"]

In [13]:
# Tenkan-sen
CBA_Stock["Tenkan_sen"] = (
    CBA_Stock["High"].rolling(9).max() +
    CBA_Stock["Low"].rolling(9).min()
) / 2

# Kijun-sen
CBA_Stock["Kijun_sen"] = (
    CBA_Stock["High"].rolling(26).max() +
    CBA_Stock["Low"].rolling(26).min()
) / 2

# NO forward shift (critical)
CBA_Stock["Senkou_A"] = (CBA_Stock["Tenkan_sen"] + CBA_Stock["Kijun_sen"]) / 2

CBA_Stock["Senkou_B"] = (
    CBA_Stock["High"].rolling(52).max() +
    CBA_Stock["Low"].rolling(52).min()
) / 2

In [14]:
CBA_Stock = CBA_Stock.dropna().reset_index(drop=True)

print("Final shape:", CBA_Stock.shape)
print("NaNs left:\n", CBA_Stock.isna().sum())

Final shape: (2803, 32)
NaNs left:
 Date              0
Open              0
High              0
Low               0
Close             0
Adj Close         0
Volume            0
Target_t1         0
Target_t7         0
Return            0
LogReturn         0
Return_lag1       0
Return_lag2       0
Return_lag3       0
Return_lag5       0
RollingMean_5     0
RollingStd_5      0
RollingMean_10    0
RollingStd_10     0
VolumeChange      0
VolumeMA_5        0
RSI_14            0
MACD              0
MACD_signal       0
BB_Mid            0
BB_Std            0
BB_Upper          0
BB_Lower          0
Tenkan_sen        0
Kijun_sen         0
Senkou_A          0
Senkou_B          0
dtype: int64


In [15]:
train_df = CBA_Stock[CBA_Stock["Date"] < "2022-01-01"]
val_df   = CBA_Stock[(CBA_Stock["Date"] >= "2022-01-01") & (CBA_Stock["Date"] < "2024-01-01")]
test_df  = CBA_Stock[CBA_Stock["Date"] >= "2024-01-01"]

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (1722, 32)
Val: (503, 32)
Test: (578, 32)


In [16]:
TARGET = "Target_t1"   # we start with t+1 first

FEATURES = [col for col in CBA_Stock.columns if col not in ["Date", "Target_t1", "Target_t7"]]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("Feature count:", len(FEATURES))

Feature count: 29


In [17]:
inf_counts = np.isinf(X_train).sum()

print("Columns with infinity:")
print(inf_counts[inf_counts > 0])


Columns with infinity:
VolumeChange    4
dtype: int64


In [18]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val   = X_val.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

# Check NaNs after replacing inf
print("Train NaNs:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

print("Val NaNs:")
print(X_val.isna().sum()[X_val.isna().sum() > 0])

print("Test NaNs:")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

Train NaNs:
VolumeChange    4
dtype: int64
Val NaNs:
Series([], dtype: int64)
Test NaNs:
Series([], dtype: int64)


In [19]:
train_valid = X_train.notna().all(axis=1)
val_valid   = X_val.notna().all(axis=1)
test_valid  = X_test.notna().all(axis=1)

X_train = X_train[train_valid]
y_train = y_train[train_valid]

X_val = X_val[val_valid]
y_val = y_val[val_valid]

X_test = X_test[test_valid]
y_test = y_test[test_valid]

print("Cleaned shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Cleaned shapes:
Train: (1718, 29) (1718,)
Val: (503, 29) (503,)
Test: (578, 29) (578,)


In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)



,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [22]:
y_val_pred = model.predict(X_val_scaled)
y_test_pred = model.predict(X_test_scaled)

y_val_prob = model.predict_proba(X_val_scaled)[:, 1]
y_test_prob = model.predict_proba(X_test_scaled)[:, 1]

In [23]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred)
val_auc = roc_auc_score(y_val, y_val_prob)

print("Validation Metrics:")
print("Accuracy:", val_accuracy)
print("F1 Score:", val_f1)
print("AUC:", val_auc)

Validation Metrics:
Accuracy: 0.5188866799204771
F1 Score: 0.675603217158177
AUC: 0.5068625586408012


In [24]:
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_prob)

print("\nTest Metrics:")
print("Accuracy:", test_accuracy)
print("F1 Score:", test_f1)
print("AUC:", test_auc)


Test Metrics:
Accuracy: 0.5640138408304498
F1 Score: 0.7129840546697038
AUC: 0.5091316690424845


In [25]:
cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[ 13 238]
 [ 14 313]]


testing it for t+7

In [26]:
TARGET = "Target_t7"   # we start with t+1 first

FEATURES = [col for col in CBA_Stock.columns if col not in ["Date", "Target_t1", "Target_t7"]]

X_train2 = train_df[FEATURES]
y_train2 = train_df[TARGET]

X_val2 = val_df[FEATURES]
y_val2 = val_df[TARGET]

X_test2 = test_df[FEATURES]
y_test2 = test_df[TARGET]

print("Feature count:", len(FEATURES))

Feature count: 29


In [27]:
inf_counts = np.isinf(X_train2).sum()

print("Columns with infinity:")
print(inf_counts[inf_counts > 0])


Columns with infinity:
VolumeChange    4
dtype: int64


In [28]:
X_train2 = X_train2.replace([np.inf, -np.inf], np.nan)
X_val2   = X_val2.replace([np.inf, -np.inf], np.nan)
X_test2  = X_test2.replace([np.inf, -np.inf], np.nan)

# Check NaNs after replacing inf
print("Train NaNs:")
print(X_train2.isna().sum()[X_train2.isna().sum() > 0])

print("Val NaNs:")
print(X_val2.isna().sum()[X_val2.isna().sum() > 0])

print("Test NaNs:")
print(X_test2.isna().sum()[X_test2.isna().sum() > 0])

Train NaNs:
VolumeChange    4
dtype: int64
Val NaNs:
Series([], dtype: int64)
Test NaNs:
Series([], dtype: int64)


In [29]:
train_valid2 = X_train2.notna().all(axis=1)
val_valid2   = X_val2.notna().all(axis=1)
test_valid2  = X_test2.notna().all(axis=1)

X_train2 = X_train2[train_valid2]
y_train2 = y_train2[train_valid2]

X_val2 = X_val2[val_valid2]
y_val2 = y_val2[val_valid2]

X_test2 = X_test2[test_valid2]
y_test2 = y_test2[test_valid2]

print("Cleaned shapes:")
print("Train:", X_train2.shape, y_train2.shape)
print("Val:", X_val2.shape, y_val2.shape)
print("Test:", X_test2.shape, y_test2.shape)

Cleaned shapes:
Train: (1718, 29) (1718,)
Val: (503, 29) (503,)
Test: (578, 29) (578,)


In [30]:
X_train_scaled2 = scaler.fit_transform(X_train2)
X_val_scaled2   = scaler.transform(X_val2)
X_test_scaled2  = scaler.transform(X_test2)

In [31]:
model.fit(X_train_scaled2, y_train2)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [32]:
y_val_pred2 = model.predict(X_val_scaled2)
y_test_pred2 = model.predict(X_test_scaled2)

y_val_prob2 = model.predict_proba(X_val_scaled2)[:, 1]
y_test_prob2 = model.predict_proba(X_test_scaled2)[:, 1]

In [33]:
val_accuracy2 = accuracy_score(y_val2, y_val_pred2)
val_f12 = f1_score(y_val2, y_val_pred2)
val_auc2 = roc_auc_score(y_val2, y_val_prob2)

print("Validation Metrics:")
print("Accuracy:", val_accuracy2)
print("F1 Score:", val_f12)
print("AUC:", val_auc2)

Validation Metrics:
Accuracy: 0.5347912524850894
F1 Score: 0.693717277486911
AUC: 0.5285714285714286


In [34]:
test_accuracy2 = accuracy_score(y_test2, y_test_pred2)
test_f12 = f1_score(y_test2, y_test_pred2)
test_auc2 = roc_auc_score(y_test2, y_test_prob2)

print("\nTest Metrics:")
print("Accuracy:", test_accuracy2)
print("F1 Score:", test_f12)
print("AUC:", test_auc2)


Test Metrics:
Accuracy: 0.596885813148789
F1 Score: 0.7459105779716467
AUC: 0.47670585308204266


In [35]:
feature_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": model.coef_[0]
})

feature_importance["Abs"] = feature_importance["Coefficient"].abs()

feature_importance = feature_importance.sort_values("Abs", ascending=False)

print(feature_importance.head(10))

        Feature  Coefficient       Abs
26    Kijun_sen    -1.054971  1.054971
20  MACD_signal    -0.553975  0.553975
3         Close     0.546256  0.546256
4     Adj Close     0.509274  0.509274
27     Senkou_A    -0.501398  0.501398
19         MACD     0.461999  0.461999
0          Open    -0.412679  0.412679
5        Volume    -0.300947  0.300947
17   VolumeMA_5    -0.244926  0.244926
18       RSI_14    -0.237083  0.237083


In [36]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)  # NOTE: use NON-scaled data

print("XGBoost trained.")

XGBoost trained.


/opt/anaconda3/envs/course/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [16:51:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [37]:
y_val_pred = xgb_model.predict(X_val)
y_test_pred = xgb_model.predict(X_test)

y_val_prob = xgb_model.predict_proba(X_val)[:, 1]
y_test_prob = xgb_model.predict_proba(X_test)[:, 1]

In [38]:
print("Validation:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("F1:", f1_score(y_val, y_val_pred))
print("AUC:", roc_auc_score(y_val, y_val_prob))

print("\nTest:")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1:", f1_score(y_test, y_test_pred))
print("AUC:", roc_auc_score(y_test, y_test_prob))

Validation:
Accuracy: 0.5506958250497018
F1: 0.6319218241042345
AUC: 0.5311430201597566

Test:
Accuracy: 0.5173010380622838
F1: 0.5181347150259067
AUC: 0.5360088697198971


In [39]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, make_scorer

tscv = TimeSeriesSplit(n_splits=5)

xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10]
}

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=40,
    scoring="roc_auc",
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best CV AUC:", search.best_score_)
print("Best Params:")
print(search.best_params_)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best CV AUC: 0.5126541846671533
Best Params:
{'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 0.01, 'n_estimators': 100, 'min_child_weight': 7, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 1.0}


In [40]:
best_xgb = search.best_estimator_

y_val_pred = best_xgb.predict(X_val)
y_test_pred = best_xgb.predict(X_test)

y_val_prob = best_xgb.predict_proba(X_val)[:, 1]
y_test_prob = best_xgb.predict_proba(X_test)[:, 1]

In [41]:
print("Validation:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("F1:", f1_score(y_val, y_val_pred))
print("AUC:", roc_auc_score(y_val, y_val_prob))

print("\nTest:")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1:", f1_score(y_test, y_test_pred))
print("AUC:", roc_auc_score(y_test, y_test_prob))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Validation:
Accuracy: 0.5089463220675944
F1: 0.5944170771756979
AUC: 0.5202548497527577

Test:
Accuracy: 0.5415224913494809
F1: 0.6324549237170597
AUC: 0.541065097408531

Test Confusion Matrix:
[[ 85 166]
 [ 99 228]]


In [42]:
import requests
import pandas as pd
from datetime import datetime, timezone

def to_utc_timestamp(date_str):
    """
    Convert YYYY-MM-DD string to UTC Unix timestamp.
    """
    return int(datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())

# Test settings
subreddit = "AusFinance"
query = "CBA"
after = to_utc_timestamp("2023-01-01")
before = to_utc_timestamp("2023-02-01")

url = "https://api.pullpush.io/reddit/search/submission/"

params = {
    "subreddit": subreddit,
    "q": query,
    "after": after,
    "before": before,
    "size": 100
}

response = requests.get(url, params=params, timeout=30)

print("Status Code:", response.status_code)
print("Final URL:", response.url)

data = response.json()
posts = data.get("data", [])

print("Number of posts collected:", len(posts))

df_reddit_test = pd.DataFrame(posts)

df_reddit_test.head()

Status Code: 200
Final URL: https://api.pullpush.io/reddit/search/submission/?subreddit=AusFinance&q=CBA&after=1672531200&before=1675209600&size=100
Number of posts collected: 27


,all_awardings,allow_live_comments,archived,author,author_created_utc,author_flair_background_color,author_flair_css_class,author_flair_richtext,author_flair_template_id,author_flair_text,...,total_awards_received,treatment_tags,upvote_ratio,url,view_count,whitelist_status,wls,url_overridden_by_dest,post_hint,preview
0,[],True,False,lambo100,1.365723e+09,None,None,[],None,None,...,0,[],0.79,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN
1,[],False,False,checkyourvitalsigns,1.568690e+09,None,None,[],None,None,...,0,[],0.33,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN
2,[],False,False,QuickBed4623,1.674639e+09,None,None,[],None,None,...,0,[],1.00,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN
3,[],False,False,Remarkable_State_540,1.636827e+09,None,None,[],None,None,...,0,[],0.60,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN
4,[],False,False,arejay007,1.514954e+09,None,None,[],None,None,...,0,[],0.30,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN


In [43]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time

def to_utc_timestamp(date_str):
    return int(datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())

def fetch_month_data(subreddit, after, before):
    url = "https://api.pullpush.io/reddit/search/submission/"
    
    params = {
        "subreddit": subreddit,
        "after": after,
        "before": before,
        "size": 100
    }
    
    response = requests.get(url, params=params, timeout=30)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        return []
    
    data = response.json()
    return data.get("data", [])

# -------- CONFIG --------
subreddits = ["AusFinance", "ASX_Bets"]
start_year = 2015
end_year = 2024   # start smaller first (we expand later)

all_posts = []

# -------- LOOP --------
for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        
        start_date = f"{year}-{month:02d}-01"
        
        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"
        
        after = to_utc_timestamp(start_date)
        before = to_utc_timestamp(end_date)
        
        print(f"Fetching {start_date} to {end_date}")
        
        for sub in subreddits:
            posts = fetch_month_data(sub, after, before)
            print(f"{sub}: {len(posts)} posts")
            
            all_posts.extend(posts)
            
            time.sleep(1)  # prevent rate issues

# -------- SAVE --------
df_reddit = pd.DataFrame(all_posts)

# Keep only important columns
df_reddit = df_reddit[[
    "created_utc",
    "subreddit",
    "title",
    "selftext",
    "score"
]]

# Save to your data folder
file_path = "data/reddit_raw.csv"
df_reddit.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Total posts:", len(df_reddit))

Fetching 2015-01-01 to 2015-02-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-02-01 to 2015-03-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-03-01 to 2015-04-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-04-01 to 2015-05-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-05-01 to 2015-06-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-06-01 to 2015-07-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-07-01 to 2015-08-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-08-01 to 2015-09-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-09-01 to 2015-10-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-10-01 to 2015-11-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-11-01 to 2015-12-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-12-01 to 2016-01-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2016-01-01 to 2016-02-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2016-02-01 to 2016-03-01
AusF

In [46]:
# -------- LOAD RAW DATA SAFELY --------
file_path = "data/reddit_raw.csv"

df_reddit_raw = pd.read_csv(
    file_path,
    engine="python",
    on_bad_lines="skip"
)

print("Raw shape after safe load:", df_reddit_raw.shape)
print(df_reddit_raw.columns)

# -------- COPY TO NEW DF --------
df_reddit_clean = df_reddit_raw.copy()

# -------- FIX created_utc --------
df_reddit_clean["created_utc"] = pd.to_numeric(
    df_reddit_clean["created_utc"],
    errors="coerce"
)

# Drop rows where created_utc is not a valid timestamp
df_reddit_clean = df_reddit_clean.dropna(subset=["created_utc"])

# Convert timestamp
df_reddit_clean["datetime"] = pd.to_datetime(
    df_reddit_clean["created_utc"],
    unit="s",
    utc=True
)

df_reddit_clean["date"] = df_reddit_clean["datetime"].dt.date

# -------- TEXT CLEANING --------
df_reddit_clean["title"] = df_reddit_clean["title"].fillna("").astype(str)
df_reddit_clean["selftext"] = df_reddit_clean["selftext"].fillna("").astype(str)

df_reddit_clean["text"] = (
    df_reddit_clean["title"] + " " + df_reddit_clean["selftext"]
).str.strip()

# Remove deleted/removed/empty posts
df_reddit_clean = df_reddit_clean[
    ~df_reddit_clean["text"].str.lower().isin(["[deleted]", "[removed]", ""])
]

# Keep natural text for FinBERT
df_reddit_clean = df_reddit_clean[[
    "date",
    "datetime",
    "subreddit",
    "text",
    "score"
]]

clean_path = "data/reddit_cleaned.csv"
df_reddit_clean.to_csv(clean_path, index=False)

print("Cleaned shape:", df_reddit_clean.shape)
print("Saved cleaned data to:", clean_path)

Raw shape after safe load: (17801, 5)
Index(['created_utc', 'subreddit', 'title', 'selftext', 'score'], dtype='object')
Cleaned shape: (17800, 5)
Saved cleaned data to: data/reddit_cleaned.csv


In [47]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

/opt/anaconda3/envs/course/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [48]:
clean_path = "data/reddit_cleaned.csv"

df_reddit_cleaned = pd.read_csv(clean_path)

print("Shape:", df_reddit_cleaned.shape)
df_reddit_cleaned.head()

Shape: (17800, 5)


,date,datetime,subreddit,text,score
0,2015-01-31,2015-01-31 16:25:46+00:00,AusFinance,Have you used OzForex? Wondering if any of you...,8.0
1,2015-01-31,2015-01-31 11:58:56+00:00,AusFinance,"I've saved $5000, what should I do with it?",2.0
2,2015-01-31,2015-01-31 04:29:33+00:00,AusFinance,How much would this hurt financially to move h...,5.0
3,2015-01-30,2015-01-30 23:00:48+00:00,AusFinance,How to Hedge the AUD against GBP I have studen...,2.0
4,2015-01-30,2015-01-30 19:26:56+00:00,AusFinance,Bank with no or low international transaction ...,2.0


In [49]:
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using device:", device)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 52494.09it/s]

Using device: cpu


In [50]:
label_map = {
    0: "positive",
    1: "negative",
    2: "neutral"
}

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    pred_class = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred_class].item()

    return label_map[pred_class], confidence

sample_texts = df_reddit_cleaned["text"].head(5)

for text in sample_texts:
    label, score = predict_sentiment(text)
    print("TEXT:", text[:150])
    print("SENTIMENT:", label, "| SCORE:", score)
    print("-" * 80)

TEXT: Have you used OzForex? Wondering if any of you have used OzForex and had could give me some customer reviews. I've read some feedback online, but neve
SENTIMENT: neutral | SCORE: 0.8356714844703674
--------------------------------------------------------------------------------
TEXT: I've saved $5000, what should I do with it?
SENTIMENT: neutral | SCORE: 0.9090133309364319
--------------------------------------------------------------------------------
TEXT: How much would this hurt financially to move house, then sell old house later? Though I could sell and move in one go, the stress is a little much. I'
SENTIMENT: neutral | SCORE: 0.8816514015197754
--------------------------------------------------------------------------------
TEXT: How to Hedge the AUD against GBP I have student debt in the UK and the fall of the AUD is a bit of a concern for me. I'm wondering how I can protect t
SENTIMENT: neutral | SCORE: 0.8794514536857605
------------------------------------------------

In [ ]:
# Read cleaned data again to keep workflow clean
clean_path = "data/reddit_cleaned.csv"
df_reddit_cleaned = pd.read_csv(clean_path)

print("Loaded:", df_reddit_cleaned.shape)

label_map = {
    0: "positive",
    1: "negative",
    2: "neutral"
}

numeric_map = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

def predict_sentiment_batch(texts, batch_size=16):
    labels = []
    confidence_scores = []

    model.eval()

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size].tolist()

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

        pred_classes = torch.argmax(probs, dim=1).cpu().numpy()
        batch_confidences = probs.max(dim=1).values.cpu().numpy()

        batch_labels = [label_map[int(cls)] for cls in pred_classes]

        labels.extend(batch_labels)
        confidence_scores.extend(batch_confidences)

    return labels, confidence_scores

Loaded: (17800, 5)


In [52]:
texts = df_reddit_cleaned["text"].fillna("").astype(str)

sentiment_labels, sentiment_scores = predict_sentiment_batch(
    texts,
    batch_size=16
)

df_reddit_sentiment = df_reddit_cleaned.copy()

df_reddit_sentiment["sentiment_label"] = sentiment_labels
df_reddit_sentiment["sentiment_confidence"] = sentiment_scores
df_reddit_sentiment["sentiment_numeric"] = df_reddit_sentiment["sentiment_label"].map(numeric_map)

print(df_reddit_sentiment[[
    "date",
    "subreddit",
    "text",
    "sentiment_label",
    "sentiment_confidence",
    "sentiment_numeric"
]].head())

100%|██████████| 1113/1113 [11:19<00:00,  1.64it/s]

         date   subreddit                                               text  \
0  2015-01-31  AusFinance  Have you used OzForex? Wondering if any of you...   
1  2015-01-31  AusFinance        I've saved $5000, what should I do with it?   
2  2015-01-31  AusFinance  How much would this hurt financially to move h...   
3  2015-01-30  AusFinance  How to Hedge the AUD against GBP I have studen...   
4  2015-01-30  AusFinance  Bank with no or low international transaction ...   

  sentiment_label  sentiment_confidence  sentiment_numeric  
0         neutral              0.835672                  0  
1         neutral              0.909013                  0  
2         neutral              0.881651                  0  
3         neutral              0.879452                  0  
4         neutral              0.576329                  0  


In [53]:
sentiment_path = "data/FinBERT_reddit_sentiment_scored.csv"

df_reddit_sentiment.to_csv(sentiment_path, index=False)

print("Saved sentiment-scored data to:", sentiment_path)
print("Shape:", df_reddit_sentiment.shape)

Saved sentiment-scored data to: data/FinBERT_reddit_sentiment_scored.csv
Shape: (17800, 8)


In [54]:
df_reddit_sentiment["sentiment_label"].value_counts(normalize=True)

sentiment_label
neutral     0.872921
negative    0.097191
positive    0.029888
Name: proportion, dtype: float64

In [55]:
file_path = "data/FinBERT_reddit_sentiment_scored.csv"

df_sentiment = pd.read_csv(file_path)

print("Loaded:", df_sentiment.shape)

# -------- GROUP BY DATE --------
daily_sentiment = df_sentiment.groupby("date").agg(
    sentiment_mean=("sentiment_numeric", "mean"),
    sentiment_std=("sentiment_numeric", "std"),
    positive_ratio=("sentiment_label", lambda x: (x == "positive").mean()),
    negative_ratio=("sentiment_label", lambda x: (x == "negative").mean()),
    post_volume=("sentiment_label", "count")
).reset_index()

# Fill NaN std (happens when only 1 post in a day)
daily_sentiment["sentiment_std"] = daily_sentiment["sentiment_std"].fillna(0)

print(daily_sentiment.head())

# -------- SAVE --------
daily_path = "data/reddit_daily_sentiment.csv"
daily_sentiment.to_csv(daily_path, index=False)

print("Saved to:", daily_path)
print("Shape:", daily_sentiment.shape)

Loaded: (17800, 8)
         date  sentiment_mean  sentiment_std  positive_ratio  negative_ratio  \
0  2015-01-08        0.000000       0.000000             0.0        0.000000   
1  2015-01-10        0.000000       0.000000             0.0        0.000000   
2  2015-01-11        0.000000       0.000000             0.0        0.000000   
3  2015-01-12       -0.111111       0.333333             0.0        0.111111   
4  2015-01-13        0.000000       0.707107             0.2        0.200000   

   post_volume  
0            6  
1            5  
2            4  
3            9  
4            5  
Saved to: data/reddit_daily_sentiment.csv
Shape: (1094, 6)


In [56]:
daily_sentiment.describe()

,sentiment_mean,sentiment_std,positive_ratio,negative_ratio,post_volume
count,1094.000000,1094.000000,1094.000000,1094.000000,1094.000000
mean,-0.059193,0.254500,0.031024,0.090217,16.270567
std,0.144171,0.217245,0.068648,0.121670,24.793307
min,-1.000000,0.000000,0.000000,0.000000,1.000000
25%,-0.125000,0.000000,0.000000,0.000000,5.000000
50%,0.000000,0.316228,0.000000,0.055301,8.000000
75%,0.000000,0.421637,0.027972,0.150000,14.000000
max,0.666667,0.816497,0.666667,1.000000,190.000000


In [58]:
stock_path = "data/CBA_target_ready_2015_present.csv"
sentiment_path = "data/reddit_daily_sentiment.csv"

stock_df = pd.read_csv(stock_path)
sentiment_df = pd.read_csv(sentiment_path)

print(stock_df.shape)
print(sentiment_df.shape)

print(stock_df.head())
print(sentiment_df.head())

(2856, 9)
(1094, 6)
         Date       Open       High        Low      Close  Adj Close   Volume  \
0  2015-01-02  84.959686  85.277962  84.661308  85.277962  51.295269   949439   
1  2015-01-05  85.238182  85.775269  85.049202  85.486832  51.420902  1351651   
2  2015-01-06  84.641411  85.337639  84.412651  84.840332  51.032032  2477275   
3  2015-01-07  84.850281  85.029312  84.094376  84.651360  50.918358  2127190   
4  2015-01-08  85.079041  85.188446  84.671249  84.929848  51.085873  1997761   

   Target_t1  Target_t7  
0          1          0  
1          0          0  
2          0          0  
3          1          0  
4          1          0  
         date  sentiment_mean  sentiment_std  positive_ratio  negative_ratio  \
0  2015-01-08        0.000000       0.000000             0.0        0.000000   
1  2015-01-10        0.000000       0.000000             0.0        0.000000   
2  2015-01-11        0.000000       0.000000             0.0        0.000000   
3  2015-01-12    

In [59]:
print(stock_df.columns)
print(sentiment_df.columns)

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume',
       'Target_t1', 'Target_t7'],
      dtype='object')
Index(['date', 'sentiment_mean', 'sentiment_std', 'positive_ratio',
       'negative_ratio', 'post_volume'],
      dtype='object')


In [61]:
stock_df["Date"] = pd.to_datetime(stock_df["Date"])
sentiment_df["date"] = pd.to_datetime(sentiment_df["date"])

In [62]:
stock_df = stock_df.rename(columns={"Date": "date"})
stock_df["date"] = pd.to_datetime(stock_df["date"])

In [63]:
stock_df = stock_df.sort_values("date").reset_index(drop=True)
sentiment_df = sentiment_df.sort_values("date").reset_index(drop=True)

In [64]:
print(stock_df[["date"]].head())
print(stock_df[["date"]].tail())

print(sentiment_df[["date"]].head())
print(sentiment_df[["date"]].tail())

        date
0 2015-01-02
1 2015-01-05
2 2015-01-06
3 2015-01-07
4 2015-01-08
           date
2851 2026-04-09
2852 2026-04-10
2853 2026-04-13
2854 2026-04-14
2855 2026-04-15
        date
0 2015-01-08
1 2015-01-10
2 2015-01-11
3 2015-01-12
4 2015-01-13
           date
1089 2024-12-27
1090 2024-12-28
1091 2024-12-29
1092 2024-12-30
1093 2024-12-31


In [66]:
merged_df = stock_df.merge(
    sentiment_df,
    on="date",
    how="left"
)

In [67]:
sentiment_cols = [
    "sentiment_mean",
    "sentiment_std",
    "positive_ratio",
    "negative_ratio",
    "post_volume"
]

merged_df[sentiment_cols] = merged_df[sentiment_cols].fillna(0)

In [68]:
merged_df = merged_df[
    (merged_df["date"] >= "2015-01-01") &
    (merged_df["date"] <= "2024-12-31")
].reset_index(drop=True)

print("Final date range:",
      merged_df["date"].min(),
      "→",
      merged_df["date"].max())

print("Shape:", merged_df.shape)

Final date range: 2015-01-02 00:00:00 → 2024-12-31 00:00:00
Shape: (2532, 14)


In [69]:
merged_df.to_csv("data/CBA_merged_sentiment_2015_2024_unshifted.csv", index=False)

In [70]:
merged_df[sentiment_cols] = merged_df[sentiment_cols].shift(1)

In [71]:
merged_df = merged_df.dropna().reset_index(drop=True)

In [72]:
merged_df.to_csv("data/CBA_final_multimodal_2015_2024.csv", index=False)

In [73]:
print(merged_df[[
    "date",
    "sentiment_mean",
    "post_volume",
    "Target_t1"
]].head(10))

        date  sentiment_mean  post_volume  Target_t1
0 2015-01-05        0.000000          0.0          0
1 2015-01-06        0.000000          0.0          0
2 2015-01-07        0.000000          0.0          1
3 2015-01-08        0.000000          0.0          1
4 2015-01-09        0.000000          6.0          0
5 2015-01-12        0.000000          0.0          0
6 2015-01-13       -0.111111          9.0          0
7 2015-01-14        0.000000          5.0          0
8 2015-01-15        0.000000          3.0          1
9 2015-01-16        0.000000          5.0          0


In [74]:
print("Any NaNs left:", merged_df.isna().sum().sum())

print("Sentiment stats:")
print(merged_df[sentiment_cols].describe())

print("Target distribution:")
print(merged_df["Target_t1"].value_counts(normalize=True))

Any NaNs left: 0
Sentiment stats:
       sentiment_mean  sentiment_std  positive_ratio  negative_ratio  \
count     2531.000000    2531.000000     2531.000000     2531.000000   
mean        -0.018632       0.083070        0.010184        0.028816   
std          0.081473       0.171290        0.039557        0.077637   
min         -1.000000       0.000000        0.000000        0.000000   
25%          0.000000       0.000000        0.000000        0.000000   
50%          0.000000       0.000000        0.000000        0.000000   
75%          0.000000       0.000000        0.000000        0.000000   
max          0.500000       0.816497        0.500000        1.000000   

       post_volume  
count  2531.000000  
mean      5.351245  
std      16.253242  
min       0.000000  
25%       0.000000  
50%       0.000000  
75%       5.000000  
max     161.000000  
Target distribution:
Target_t1
1    0.522718
0    0.477282
Name: proportion, dtype: float64


In [84]:
features = [col for col in merged_df.columns
            if col not in ["date", "Target_t1", "Target_t7"]]

X = merged_df[features]
y = merged_df["Target_t1"]

In [85]:
split = int(len(merged_df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [86]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

print("Logistic Regression Results (WITH SENTIMENT)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))

Logistic Regression Results (WITH SENTIMENT)
Accuracy: 0.5652173913043478
F1: 0.7201017811704835
AUC: 0.5308486967577877


In [87]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("\nXGBoost Results (WITH SENTIMENT)")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1:", f1_score(y_test, y_pred_xgb))
print("AUC:", roc_auc_score(y_test, y_prob_xgb))


XGBoost Results (WITH SENTIMENT)
Accuracy: 0.48616600790513836
F1: 0.38095238095238093
AUC: 0.49009059122695486


In [79]:
for lag in [1, 2, 3]:
    merged_df[f"sentiment_mean_lag_{lag}"] = merged_df["sentiment_mean"].shift(lag)

In [80]:
merged_df["sentiment_rolling_mean_3"] = merged_df["sentiment_mean"].rolling(3).mean()
merged_df["sentiment_rolling_std_3"] = merged_df["sentiment_mean"].rolling(3).std()

In [81]:
merged_df["sentiment_change"] = merged_df["sentiment_mean"].diff()

In [82]:
merged_df["sentiment_weighted"] = merged_df["sentiment_mean"] * merged_df["post_volume"]

In [83]:
merged_df = merged_df.dropna().reset_index(drop=True)

CLEAN LSTM PIPELINE
- WITH STOCK DATA
- WITH MERGED DATA

In [125]:
# =========================
# CLEAN LSTM PIPELINE
# Stock-only first, then stock + sentiment
# =========================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# -------------------------
# 1. Reload clean data
# -------------------------

stock_path = "data/CBA_target_ready_2015_present.csv"
sentiment_path = "data/reddit_daily_sentiment.csv"

stock_df = pd.read_csv(stock_path)
sentiment_df = pd.read_csv(sentiment_path)

# Fix date column safely
if "Date" in stock_df.columns:
    stock_df["date"] = pd.to_datetime(stock_df["Date"])
else:
    stock_df["date"] = pd.to_datetime(stock_df["date"])

sentiment_df["date"] = pd.to_datetime(sentiment_df["date"])

# Restrict stock to 2015–2024
stock_df = stock_df[
    (stock_df["date"] >= "2015-01-01") &
    (stock_df["date"] <= "2024-12-31")
].reset_index(drop=True)

# Merge sentiment
merged_df = stock_df.merge(sentiment_df, on="date", how="left")

sentiment_cols = [
    "sentiment_mean",
    "sentiment_std",
    "positive_ratio",
    "negative_ratio",
    "post_volume"
]

for col in sentiment_cols:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].fillna(0)

# Leakage prevention: sentiment from previous day only
for col in sentiment_cols:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].shift(1)

merged_df = merged_df.dropna().reset_index(drop=True)

print("Merged shape:", merged_df.shape)
print("Date range:", merged_df["date"].min(), "→", merged_df["date"].max())
print("Target balance:")
print(merged_df["Target_t1"].value_counts(normalize=True))

Merged shape: (2531, 15)
Date range: 2015-01-05 00:00:00 → 2024-12-31 00:00:00
Target balance:
Target_t1
1    0.522718
0    0.477282
Name: proportion, dtype: float64


In [126]:
# -------------------------
# 2. Feature sets
# -------------------------

target_col = "Target_t1"

exclude_common = ["date", "Date", "Target_t1", "Target_t7"]

sentiment_related_cols = [
    col for col in merged_df.columns
    if "sentiment" in col.lower()
    or "positive_ratio" in col.lower()
    or "negative_ratio" in col.lower()
    or "post_volume" in col.lower()
]

stock_features = [
    col for col in merged_df.columns
    if col not in exclude_common + sentiment_related_cols
]

stock_features = merged_df[stock_features].select_dtypes(include=[np.number]).columns.tolist()

print("Number of stock-only features:", len(stock_features))
print(stock_features)

Number of stock-only features: 6
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']


In [127]:
# -------------------------
# 3. Sequence creation
# -------------------------

def create_sequences(X, y, seq_len=30):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


def prepare_lstm_data(df, feature_cols, target_col="Target_t1", seq_len=30):
    X = df[feature_cols].values.astype(np.float32)
    y = df[target_col].values.astype(np.float32)

    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    X_train_raw = X[:train_end]
    y_train_raw = y[:train_end]

    X_val_raw = X[train_end:val_end]
    y_val_raw = y[train_end:val_end]

    X_test_raw = X[val_end:]
    y_test_raw = y[val_end:]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_val_scaled = scaler.transform(X_val_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_raw, seq_len)
    X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val_raw, seq_len)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_raw, seq_len)

    return X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_test_seq, y_test_seq

In [128]:
# -------------------------
# 4. LSTM model
# -------------------------

class StockLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        return self.fc(last_hidden).squeeze()

In [129]:
# -------------------------
# 5. Training function
# -------------------------

def train_and_evaluate_lstm(df, feature_cols, experiment_name, epochs=100, lr=0.001, seq_len=30):
    print("\n==============================")
    print(experiment_name)
    print("==============================")

    X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_test_seq, y_test_seq = prepare_lstm_data(
        df, feature_cols, target_col="Target_t1", seq_len=seq_len
    )

    print("Train:", X_train_seq.shape, y_train_seq.shape)
    print("Val:", X_val_seq.shape, y_val_seq.shape)
    print("Test:", X_test_seq.shape, y_test_seq.shape)

    X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_seq, dtype=torch.float32)

    X_val_t = torch.tensor(X_val_seq, dtype=torch.float32)
    y_val_t = torch.tensor(y_val_seq, dtype=torch.float32)

    X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)

    input_size = X_train_seq.shape[2]

    model = StockLSTM(input_size=input_size)

    positive_count = np.sum(y_train_seq == 1)
    negative_count = np.sum(y_train_seq == 0)

    pos_weight_value = negative_count / positive_count
    pos_weight = torch.tensor(pos_weight_value, dtype=torch.float32)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_auc = -1
    best_state = None

    for epoch in range(epochs):
        model.train()

        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs = torch.sigmoid(val_logits).numpy()

            try:
                val_auc = roc_auc_score(y_val_seq, val_probs)
            except:
                val_auc = 0.5

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {loss.item():.4f} | Val AUC: {val_auc:.4f}")

    model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        probs = torch.sigmoid(test_logits).numpy()

    y_pred = (probs >= 0.5).astype(int)

    print("\nFinal Test Results")
    print("Accuracy:", accuracy_score(y_test_seq, y_pred))
    print("F1:", f1_score(y_test_seq, y_pred))
    print("AUC:", roc_auc_score(y_test_seq, probs))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_seq, y_pred))

    print("\nPredicted class distribution:")
    print(np.unique(y_pred, return_counts=True))

    print("\nProbability range:")
    print(probs.min(), probs.max(), probs.mean())

    print("\nClassification Report:")
    print(classification_report(y_test_seq, y_pred))

    return model, probs, y_pred, y_test_seq

In [130]:
# -------------------------
# 6. Run stock-only LSTM
# -------------------------

stock_model, stock_probs, stock_pred, stock_y_test = train_and_evaluate_lstm(
    merged_df,
    stock_features,
    experiment_name="STOCK-ONLY LSTM",
    epochs=100,
    lr=0.001,
    seq_len=30
)


STOCK-ONLY LSTM
Train: (1741, 30, 6) (1741,)
Val: (350, 30, 6) (350,)
Test: (350, 30, 6) (350,)
Epoch 10/100 | Train Loss: 0.6767 | Val AUC: 0.5322
Epoch 20/100 | Train Loss: 0.6766 | Val AUC: 0.4969
Epoch 30/100 | Train Loss: 0.6757 | Val AUC: 0.4522
Epoch 40/100 | Train Loss: 0.6745 | Val AUC: 0.4365
Epoch 50/100 | Train Loss: 0.6739 | Val AUC: 0.4308
Epoch 60/100 | Train Loss: 0.6734 | Val AUC: 0.4337
Epoch 70/100 | Train Loss: 0.6735 | Val AUC: 0.4606
Epoch 80/100 | Train Loss: 0.6729 | Val AUC: 0.5151
Epoch 90/100 | Train Loss: 0.6733 | Val AUC: 0.5399
Epoch 100/100 | Train Loss: 0.6707 | Val AUC: 0.5609

Final Test Results
Accuracy: 0.5742857142857143
F1: 0.7276051188299817
AUC: 0.4997993042547498

Confusion Matrix:
[[  2 146]
 [  3 199]]

Predicted class distribution:
(array([0, 1]), array([  5, 345]))

Probability range:
0.47268614 0.56362915 0.5431607

Classification Report:
              precision    recall  f1-score   support

         0.0       0.40      0.01      0.03    

In [131]:
# -------------------------
# 7. Stock + sentiment feature set
# -------------------------

merged_features = [
    col for col in merged_df.columns
    if col not in exclude_common
]

merged_features = merged_df[merged_features].select_dtypes(include=[np.number]).columns.tolist()

print("Number of merged features:", len(merged_features))
print(merged_features)

Number of merged features: 11
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'sentiment_mean', 'sentiment_std', 'positive_ratio', 'negative_ratio', 'post_volume']


In [132]:
# -------------------------
# 8. Run stock + sentiment LSTM
# -------------------------

merged_model, merged_probs, merged_pred, merged_y_test = train_and_evaluate_lstm(
    merged_df,
    merged_features,
    experiment_name="STOCK + SENTIMENT LSTM",
    epochs=100,
    lr=0.001,
    seq_len=30
)


STOCK + SENTIMENT LSTM
Train: (1741, 30, 11) (1741,)
Val: (350, 30, 11) (350,)
Test: (350, 30, 11) (350,)
Epoch 10/100 | Train Loss: 0.6772 | Val AUC: 0.4716
Epoch 20/100 | Train Loss: 0.6758 | Val AUC: 0.4504
Epoch 30/100 | Train Loss: 0.6747 | Val AUC: 0.4319
Epoch 40/100 | Train Loss: 0.6725 | Val AUC: 0.4306
Epoch 50/100 | Train Loss: 0.6709 | Val AUC: 0.4702
Epoch 60/100 | Train Loss: 0.6675 | Val AUC: 0.5298
Epoch 70/100 | Train Loss: 0.6637 | Val AUC: 0.5369
Epoch 80/100 | Train Loss: 0.6602 | Val AUC: 0.5334
Epoch 90/100 | Train Loss: 0.6579 | Val AUC: 0.5374
Epoch 100/100 | Train Loss: 0.6527 | Val AUC: 0.5437

Final Test Results
Accuracy: 0.4742857142857143
F1: 0.42138364779874216
AUC: 0.5053518865400054

Confusion Matrix:
[[ 99  49]
 [135  67]]

Predicted class distribution:
(array([0, 1]), array([234, 116]))

Probability range:
0.3577417 0.7623908 0.4657775

Classification Report:
              precision    recall  f1-score   support

         0.0       0.42      0.67     

NEW CLEAN PIPELINE FOR TARGET + 7 DAYS

In [133]:
# ============================================================
# TARGET_t7 CLEAN PIPELINE: Baseline + Improved Deep Learning
# ============================================================

# -----------------------------
# 1. Load fresh data
# -----------------------------

stock_path = "data/CBA_target_ready_2015_present.csv"
sentiment_path = "data/reddit_daily_sentiment.csv"

stock_df = pd.read_csv(stock_path)
sentiment_df = pd.read_csv(sentiment_path)

if "Date" in stock_df.columns:
    stock_df["date"] = pd.to_datetime(stock_df["Date"])
else:
    stock_df["date"] = pd.to_datetime(stock_df["date"])

sentiment_df["date"] = pd.to_datetime(sentiment_df["date"])

stock_df = stock_df[
    (stock_df["date"] >= "2015-01-01") &
    (stock_df["date"] <= "2024-12-31")
].reset_index(drop=True)

merged_df = stock_df.merge(sentiment_df, on="date", how="left")

sentiment_cols = [
    "sentiment_mean",
    "sentiment_std",
    "positive_ratio",
    "negative_ratio",
    "post_volume"
]

for col in sentiment_cols:
    merged_df[col] = merged_df[col].fillna(0)

# leakage prevention
for col in sentiment_cols:
    merged_df[col] = merged_df[col].shift(1)

merged_df = merged_df.dropna().reset_index(drop=True)

target_col = "Target_t7"

print("Shape:", merged_df.shape)
print("Date range:", merged_df["date"].min(), "→", merged_df["date"].max())
print("Target balance:")
print(merged_df[target_col].value_counts(normalize=True))

Shape: (2531, 15)
Date range: 2015-01-05 00:00:00 → 2024-12-31 00:00:00
Target balance:
Target_t7
1    0.549585
0    0.450415
Name: proportion, dtype: float64


In [134]:
# -----------------------------
# 2. Feature selection
# -----------------------------

exclude_cols = ["date", "Date", "Target_t1", "Target_t7"]

all_features = [
    col for col in merged_df.columns
    if col not in exclude_cols
]

all_features = merged_df[all_features].select_dtypes(include=[np.number]).columns.tolist()

sentiment_related_cols = [
    col for col in all_features
    if "sentiment" in col.lower()
    or "positive_ratio" in col.lower()
    or "negative_ratio" in col.lower()
    or "post_volume" in col.lower()
]

stock_features = [
    col for col in all_features
    if col not in sentiment_related_cols
]

merged_features = all_features

print("Stock-only features:", len(stock_features))
print(stock_features)

print("\nStock + sentiment features:", len(merged_features))
print(merged_features)

Stock-only features: 6
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

Stock + sentiment features: 11
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'sentiment_mean', 'sentiment_std', 'positive_ratio', 'negative_ratio', 'post_volume']


In [135]:
# -----------------------------
# 3. Tabular baseline function
# -----------------------------

def run_tabular_baselines(df, feature_cols, target_col, experiment_name):
    print("\n==============================")
    print(experiment_name)
    print("==============================")

    X = df[feature_cols].values
    y = df[target_col].values

    split = int(len(df) * 0.8)

    X_train_raw, X_test_raw = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    lr = LogisticRegression(max_iter=2000, class_weight="balanced")
    lr.fit(X_train, y_train)

    lr_pred = lr.predict(X_test)
    lr_prob = lr.predict_proba(X_test)[:, 1]

    print("\nLogistic Regression")
    print("Accuracy:", accuracy_score(y_test, lr_pred))
    print("F1:", f1_score(y_test, lr_pred))
    print("AUC:", roc_auc_score(y_test, lr_prob))
    print(confusion_matrix(y_test, lr_pred))

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

    xgb.fit(X_train, y_train)

    xgb_pred = xgb.predict(X_test)
    xgb_prob = xgb.predict_proba(X_test)[:, 1]

    print("\nXGBoost")
    print("Accuracy:", accuracy_score(y_test, xgb_pred))
    print("F1:", f1_score(y_test, xgb_pred))
    print("AUC:", roc_auc_score(y_test, xgb_prob))
    print(confusion_matrix(y_test, xgb_pred))

In [136]:
# -----------------------------
# 4. Run Target_t7 baselines
# -----------------------------

run_tabular_baselines(
    merged_df,
    stock_features,
    target_col,
    "TARGET_t7 BASELINE: STOCK ONLY"
)

run_tabular_baselines(
    merged_df,
    merged_features,
    target_col,
    "TARGET_t7 BASELINE: STOCK + SENTIMENT"
)


TARGET_t7 BASELINE: STOCK ONLY

Logistic Regression
Accuracy: 0.5167652859960552
F1: 0.608
AUC: 0.5292100205447677
[[ 72 119]
 [126 190]]

XGBoost
Accuracy: 0.40236686390532544
F1: 0.3700623700623701
AUC: 0.43542481277752
[[115  76]
 [227  89]]

TARGET_t7 BASELINE: STOCK + SENTIMENT

Logistic Regression
Accuracy: 0.5088757396449705
F1: 0.5414364640883977
AUC: 0.5223175823447546
[[111  80]
 [169 147]]

XGBoost
Accuracy: 0.4260355029585799
F1: 0.37687366167023556
AUC: 0.4538405460931805
[[128  63]
 [228  88]]


In [137]:
# ============================================================
# IMPROVED LSTM FOR Target_t7
# ============================================================

import copy
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEQ_LEN = 60
BATCH_SIZE = 32
EPOCHS = 150
LR = 0.0003
target_col = "Target_t7"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [138]:
def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


def prepare_sequence_data(df, feature_cols, target_col, seq_len=60):
    X = df[feature_cols].values.astype(np.float32)
    y = df[target_col].values.astype(np.float32)

    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    X_train_raw, y_train_raw = X[:train_end], y[:train_end]
    X_val_raw, y_val_raw = X[train_end:val_end], y[train_end:val_end]
    X_test_raw, y_test_raw = X[val_end:], y[val_end:]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    X_test = scaler.transform(X_test_raw)

    X_train_seq, y_train_seq = create_sequences(X_train, y_train_raw, seq_len)
    X_val_seq, y_val_seq = create_sequences(X_val, y_val_raw, seq_len)
    X_test_seq, y_test_seq = create_sequences(X_test, y_test_raw, seq_len)

    return X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_test_seq, y_test_seq

In [139]:
class ImprovedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=96, num_layers=2, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=False
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(1)

In [140]:
def find_best_threshold(y_true, probs):
    thresholds = np.arange(0.30, 0.71, 0.01)
    best_t = 0.5
    best_f1 = -1

    for t in thresholds:
        preds = (probs >= t).astype(int)
        score = f1_score(y_true, preds)

        if score > best_f1:
            best_f1 = score
            best_t = t

    return best_t, best_f1

In [141]:
def train_lstm_model(df, feature_cols, experiment_name):
    print("\n==============================")
    print(experiment_name)
    print("==============================")

    X_train, y_train, X_val, y_val, X_test, y_test = prepare_sequence_data(
        df, feature_cols, target_col, SEQ_LEN
    )

    print("Train:", X_train.shape, y_train.shape)
    print("Val:", X_val.shape, y_val.shape)
    print("Test:", X_test.shape, y_test.shape)

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32)
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

    model = ImprovedLSTM(input_size=X_train.shape[2]).to(device)

    pos_count = np.sum(y_train == 1)
    neg_count = np.sum(y_train == 0)
    pos_weight = torch.tensor(neg_count / pos_count, dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    best_auc = -1
    best_state = None
    patience = 25
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        epoch_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()

        val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{EPOCHS} | "
                f"Loss: {np.mean(epoch_losses):.4f} | "
                f"Val AUC: {val_auc:.4f} | "
                f"Best Val AUC: {best_auc:.4f}"
            )

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(model(X_val_t)).cpu().numpy()
        test_probs = torch.sigmoid(model(X_test_t)).cpu().numpy()

    best_threshold, best_val_f1 = find_best_threshold(y_val, val_probs)

    test_pred_default = (test_probs >= 0.5).astype(int)
    test_pred_tuned = (test_probs >= best_threshold).astype(int)

    print("\nBest validation threshold:", best_threshold)
    print("Best validation F1:", best_val_f1)

    print("\nFinal Test Results — Default Threshold 0.50")
    print("Accuracy:", accuracy_score(y_test, test_pred_default))
    print("F1:", f1_score(y_test, test_pred_default))
    print("AUC:", roc_auc_score(y_test, test_probs))
    print(confusion_matrix(y_test, test_pred_default))

    print("\nFinal Test Results — Tuned Threshold")
    print("Accuracy:", accuracy_score(y_test, test_pred_tuned))
    print("F1:", f1_score(y_test, test_pred_tuned))
    print("AUC:", roc_auc_score(y_test, test_probs))
    print(confusion_matrix(y_test, test_pred_tuned))

    print("\nPrediction distribution:")
    print(np.unique(test_pred_tuned, return_counts=True))

    print("\nProbability range:")
    print(test_probs.min(), test_probs.max(), test_probs.mean())

    return model, test_probs, test_pred_tuned, y_test

In [142]:
lstm_stock_model, lstm_stock_probs, lstm_stock_pred, lstm_stock_y = train_lstm_model(
    merged_df,
    stock_features,
    "IMPROVED LSTM Target_t7 — STOCK ONLY"
)


IMPROVED LSTM Target_t7 — STOCK ONLY
Train: (1711, 60, 6) (1711,)
Val: (320, 60, 6) (320,)
Test: (320, 60, 6) (320,)
Epoch 10/150 | Loss: 0.6298 | Val AUC: 0.6429 | Best Val AUC: 0.6609
Epoch 20/150 | Loss: 0.6198 | Val AUC: 0.6358 | Best Val AUC: 0.6609
Epoch 30/150 | Loss: 0.6196 | Val AUC: 0.6725 | Best Val AUC: 0.6807
Epoch 40/150 | Loss: 0.5987 | Val AUC: 0.6800 | Best Val AUC: 0.6857
Epoch 50/150 | Loss: 0.5701 | Val AUC: 0.6848 | Best Val AUC: 0.6867
Epoch 60/150 | Loss: 0.5428 | Val AUC: 0.6582 | Best Val AUC: 0.6867
Epoch 70/150 | Loss: 0.5177 | Val AUC: 0.6356 | Best Val AUC: 0.6867
Early stopping at epoch 73

Best validation threshold: 0.39000000000000007
Best validation F1: 0.7268408551068883

Final Test Results — Default Threshold 0.50
Accuracy: 0.384375
F1: 0.2730627306273063
AUC: 0.5735287319125331
[[ 86  11]
 [186  37]]

Final Test Results — Tuned Threshold
Accuracy: 0.415625
F1: 0.33451957295373663
AUC: 0.5735287319125331
[[ 86  11]
 [176  47]]

Prediction distributio

In [143]:
lstm_merged_model, lstm_merged_probs, lstm_merged_pred, lstm_merged_y = train_lstm_model(
    merged_df,
    merged_features,
    "IMPROVED LSTM Target_t7 — STOCK + SENTIMENT"
)


IMPROVED LSTM Target_t7 — STOCK + SENTIMENT
Train: (1711, 60, 11) (1711,)
Val: (320, 60, 11) (320,)
Test: (320, 60, 11) (320,)
Epoch 10/150 | Loss: 0.6219 | Val AUC: 0.6557 | Best Val AUC: 0.6623
Epoch 20/150 | Loss: 0.5985 | Val AUC: 0.5712 | Best Val AUC: 0.6623
Epoch 30/150 | Loss: 0.5439 | Val AUC: 0.5426 | Best Val AUC: 0.6623
Early stopping at epoch 32

Best validation threshold: 0.6200000000000003
Best validation F1: 0.6946902654867256

Final Test Results — Default Threshold 0.50
Accuracy: 0.696875
F1: 0.8213627992633518
AUC: 0.5896167537330683
[[  0  97]
 [  0 223]]

Final Test Results — Tuned Threshold
Accuracy: 0.528125
F1: 0.5698005698005698
AUC: 0.5896167537330683
[[ 69  28]
 [123 100]]

Prediction distribution:
(array([0, 1]), array([192, 128]))

Probability range:
0.5548236 0.67447805 0.6126156
